In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import pandas as pd
from src.donnees import charger_raca, nettoyer_raca, reechantillonner_raca
from src.cibles import ajouter_cible
from src.evaluation import scores_regression

serie = reechantillonner_raca(nettoyer_raca(charger_raca()))
serie.shape

(200630, 21)

In [2]:
lignes = []
for horizon in [1, 5, 10]:
    df = ajouter_cible(serie, horizon)
    for application in ["Download", "Netflix", "Amazon_Prime"]:
        sel = df[df["evaluable"] & (df["application"] == application)]
        scores = scores_regression(sel["cible"], sel["debit"])    # prédiction = débit actuel
        lignes.append({"horizon_s": horizon, "application": application, "n": len(sel),
                       "debit_moyen": sel["cible"].mean(), **scores})

resultats = pd.DataFrame(lignes).round(2)
resultats

,horizon_s,application,n,debit_moyen,MAE,RMSE,R2
0,1,Download,38912,44.98,15.27,35.85,0.73
1,1,Netflix,70921,0.76,0.95,4.59,-0.25
2,1,Amazon_Prime,70822,2.42,3.50,6.34,-0.12
3,5,Download,38838,45.06,35.44,69.05,0.01
4,5,Netflix,70801,0.76,1.33,5.58,-0.85
5,5,Amazon_Prime,70716,2.43,2.64,7.04,-0.38
6,10,Download,38745,45.11,39.16,73.34,-0.11
7,10,Netflix,70645,0.74,1.39,5.74,-1.00
8,10,Amazon_Prime,70582,2.42,2.75,7.48,-0.57


In [3]:
resultats["MAE_relative"] = (resultats["MAE"] / resultats["debit_moyen"]).round(2)
resultats.to_csv("../results/tables/reference_persistance.csv", index=False)
resultats

,horizon_s,application,n,debit_moyen,MAE,RMSE,R2,MAE_relative
0,1,Download,38912,44.98,15.27,35.85,0.73,0.34
1,1,Netflix,70921,0.76,0.95,4.59,-0.25,1.25
2,1,Amazon_Prime,70822,2.42,3.50,6.34,-0.12,1.45
3,5,Download,38838,45.06,35.44,69.05,0.01,0.79
4,5,Netflix,70801,0.76,1.33,5.58,-0.85,1.75
5,5,Amazon_Prime,70716,2.43,2.64,7.04,-0.38,1.09
6,10,Download,38745,45.11,39.16,73.34,-0.11,0.87
7,10,Netflix,70645,0.74,1.39,5.74,-1.00,1.88
8,10,Amazon_Prime,70582,2.42,2.75,7.48,-0.57,1.14
